# First image inside GPU JupyterLab

Sana reference path. Other model families require their own pipeline and precision recipe. Start from a CUDA-enabled PyTorch GPU environment; record the resource creation/RUNNING timestamps separately.

In [ ]:
from pathlib import Path
import sys

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "generation.json").is_file())
sys.path.insert(0, str(CODE.resolve()))


## Prepare the live runtime
This cell installs the pinned non-PyTorch libraries. Use a compatible CUDA PyTorch base (the code/Dockerfile records the reference). Restart the kernel if libraries were already imported, then run all cells again.

In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "diffusers==0.39.0", "transformers==5.13.1", "accelerate==1.14.0", "sentencepiece==0.2.1", "protobuf==6.33.0"])
import torch
assert torch.cuda.is_available(), "Select a GPU runtime with CUDA-enabled torch"
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())

## Load once and generate
The worker synchronizes GPU timers, resets the same seed for each independent generation and writes configuration, runtime metadata and peak allocated memory. Loading includes a weight download on an empty cache.

In [ ]:
from generate import run
folder = run(CODE / "generation.json", n=3, mode="jupyterlab")

In [ ]:
import json
report = json.loads((folder / "metrics.json").read_text())
print(json.dumps(report, indent=2))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(folder / "asset-000.png")))

## One controlled change
Copy generation.json, change only the seed or inference-step count, and pass the copy to run(). Explain what changed. This is optional paid work; do not run a sweep until you know one asset cost.

## Download and clean up
Download PNGs, metrics.json and trace.jsonl before stopping the Devlab. Stop or delete it in the console, confirm state, and record the timestamp. A stopped Devlab may retain a billable disk; delete what you no longer need after retaining outputs.

In [ ]:
print("ACTION: download your results, stop/delete the Devlab, and verify resource state.")